# Preprocessing of the Data. Download, import and get a first overview at the Data.

### Packages einladen

In [1]:
# import alle the needed packages
import os
import sys
import time
import json
import numpy as np
import pandas as pd
import glob
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import sklearn
import os
from dotenv import load_dotenv
from pathlib import Path


In [2]:
# Working Directory prüfen
cwd = Path.cwd()
print(f"Working Directory: {cwd}")

# Versuche .env im aktuellen oder übergeordneten Verzeichnis zu finden
env_path = None
for possible_path in [Path(".env"), Path("../.env"), cwd / ".env", cwd.parent / ".env"]:
    if possible_path.exists():
        env_path = possible_path
        print(f"✓ .env gefunden at: {env_path}")
        break

if env_path:
    load_dotenv(dotenv_path=env_path)
else:
    print("✗ Keine .env Datei gefunden!")

print("Data_path:", os.getenv("Data_path"))

data_base_path = os.getenv("Data_path")



Working Directory: c:\Users\User\Documents\Uni\Master_Bauing\WiSe25_26\AI_in_Human_Water\berlin-green-roofs\notebooks
✓ .env gefunden at: ..\.env
Data_path: C:\\Users\\User\\Documents\\Uni\\Master_Bauing\\WiSe25_26\\AI_in_Human_Water\\berlin-green-roofs\\data


In [3]:
# load shape file
shape_file_path = Path(data_base_path) / "green roofs 2020.shp"
shapefile = gpd.read_file(shape_file_path)



In [4]:
# Konvertieren in pandas  Dataframe um eine attribute Tabelle zu bekommen
attribute_table = pd.DataFrame(shapefile.drop(columns="geometry"))
print(attribute_table.head())


         gml_id  importid    geb_nutz        gruendach    ex_int  gruen20_m2  \
0  d_gebaeude.1         1  Tiefgarage        vorhanden  intensiv      210.09   
1  d_gebaeude.2         2  Tiefgarage        vorhanden  intensiv      559.36   
2  d_gebaeude.3         3  Tiefgarage  nicht vorhanden       NaN        0.00   
3  d_gebaeude.4         4  Tiefgarage        vorhanden  intensiv      567.69   
4  d_gebaeude.5         5  Tiefgarage  nicht vorhanden       NaN        0.00   

   gint20_m2  gex20_m2  gruen20_p  gint20_p  gex20_p  geb_area  \
0     207.88      2.21      86.03     85.13     0.90    244.19   
1     480.07     79.29      61.59     52.86     8.73    908.26   
2       0.00      0.00       0.00      0.00     0.00    435.39   
3     364.62    203.08      65.13     41.83    23.30    871.58   
4       0.00      0.00       0.00      0.00     0.00     27.12   

                 nutz  ext            egeb_nutz    egruendach    eex_int  
0  Tiefgarage (ALKIS)    0  Underground parking

In [5]:
# lade die geojson datei ein
geojson_file_path = Path(data_base_path) / "export.geojson"

# wandle die datei in einen pandas dataframe umeine attribute Tabelle zu bekommen
geojson_data = gpd.read_file(geojson_file_path)
print(geojson_data.head())

Skipping field check_out: unsupported OGR type: 10


            id          @id 3dr:height1 3dr:height2 3dr:length1 3dr:length2  \
0  way/4675901  way/4675901         NaN         NaN         NaN         NaN   
1  way/4676147  way/4676147         NaN         NaN         NaN         NaN   
2  way/4703544  way/4703544         NaN         NaN         NaN         NaN   
3  way/4706588  way/4706588         NaN         NaN         NaN         NaN   
4  way/4706590  way/4706590         NaN         NaN         NaN         NaN   

  3dr:type abandoned abandoned:amenity abandoned:building  ...  \
0      NaN       NaN               NaN                NaN  ...   
1      NaN       NaN               NaN                NaN  ...   
2      NaN       NaN               NaN                NaN  ...   
3      NaN       NaN               NaN                NaN  ...   
4      NaN       NaN               NaN                NaN  ...   

                                   wikimedia_commons  \
0                                                NaN   
1               

In [5]:
# lade die Daten über die Geschosszahl ein
floors_shapes_file_path = Path(data_base_path) / "Geschosszahl.shp"
floors_data = gpd.read_file(floors_shapes_file_path) 


In [9]:
# wandle die Daten über die Geschosszahl in einen pandas dataframe um eine attribute Tabelle zu bekommen
floors_attribute_table = pd.DataFrame(floors_data.drop(columns="geometry"))
print(floors_attribute_table.head())
floors_data = floors_data[["geometry", "aog", "aug"]]
shapefile = shapefile[["geometry", "ex_int"]]
# wandle die spalte "ex_int" in eine numerische Spalte um. 
# Die Werte in der Spalte "ex_int" sind entweder "extensiv" oder "intensiv" oder nicht vorhanden.
shapefile["ex_int"] = shapefile["ex_int"].map({"extensiv": 1, "intensiv": 0, np.nan: 0})

   aog  aug
0  1.0  NaN
1  1.0  NaN
2  2.0  NaN
3  3.0  NaN
4  1.0  NaN


In [10]:
# Green Roofs mit Geschosszahl anreichern
green_roofs_with_floors = gpd.sjoin(
    shapefile,                    # Linker Layer (Green Roofs)
    floors_data,                  # Rechter Layer (Geschosszahl)
    how="left",                   # left=alle Green Roofs behalten
    predicate="intersects"        # Geometrien schneiden sich
)

In [16]:
# wandle Greenroofs with floors and districts in einen pandas dataframe um eine attribute Tabelle zu bekommen
green_roofs_districts_floors_attribute_table = pd.DataFrame(green_roofs_with_floors.drop(columns="geometry"))
print(green_roofs_districts_floors_attribute_table.head())
# werfe die spalte "index_right" weg, da sie nicht mehr benötigt wird
green_roofs_districts_floors_attribute_table = green_roofs_districts_floors_attribute_table.drop(columns=["index_right"])
green_roofs_with_floors = green_roofs_with_floors.drop(columns=["index_right"])

   ex_int  index_right  aog  aug
0       0     277022.0  3.0  NaN
0       0     926111.0  3.0  NaN
0       0     676270.0  NaN  1.0
0       0      27181.0  NaN  1.0
1       0      29377.0  4.0  NaN


In [ ]:
# lade das solarpotential ein
# solarpotential einladen
solar_potential_file_path = Path(data_base_path) / "solarpotential.shp"
solar_potential_data = gpd.read_file(solar_potential_file_path)

# wandle die Daten über das solarpotential in einen pandas dataframe um eine attribute Tabelle zu bekommen
solar_potential_attribute_table = pd.DataFrame(solar_potential_data.drop(columns="geometry"))
print(solar_potential_attribute_table.head())
# lösche alle unnötigen spalten bzw behalte nur die spalten "geometry" und "neigung"
solar_potential_data = solar_potential_data[["geometry", "neigung"]]


Empty DataFrame
Columns: []
Index: [0, 1, 2, 3, 4]


In [ ]:
# lade die Daten über die Bezirke ein:
districts_shapes_file_path = Path(data_base_path) / "Bezirke.shp"
districts_data = gpd.read_file(districts_shapes_file_path)

# wandle die Daten über die Bezirke in einen pandas dataframe um eine attribute Tabelle zu bekommen
districts_attribute_table = pd.DataFrame(districts_data.drop(columns="geometry"))
print(districts_attribute_table.head())



# Green Roofs mit Bezirken anreichern
green_roofs_districts_floors = gpd.sjoin(
    green_roofs_with_floors,      # Linker Layer (Green Roofs)
    districts_data,               # Rechter Layer (Bezirke)
    how="left",                   # left=alle Green Roofs behalten
    predicate="intersects"        # Geometrien schneiden sich
)
# # Green Roofs mit solarpotential anreichern
# green_roofs_with_solar = gpd.sjoin(
#     shapefile,                    # Linker Layer (Green Roofs)
#     solar_potential_data,         # Rechter Layer (Solarpotential)
#     how="left",                   # left=alle Green Roofs behalten
#     predicate="intersects"        # Geometrien schneiden sich
#)

                    gml_id      name  gem                      namgem  namlan  \
0  bezirksgrenzen.11000001  11000001  001                       Mitte  Berlin   
1  bezirksgrenzen.11000002  11000002  002    Friedrichshain-Kreuzberg  Berlin   
2  bezirksgrenzen.11000003  11000003  003                      Pankow  Berlin   
3  bezirksgrenzen.11000004  11000004  004  Charlottenburg-Wilmersdorf  Berlin   
4  bezirksgrenzen.11000005  11000005  005                     Spandau  Berlin   

  lan  
0  11  
1  11  
2  11  
3  11  
4  11  


In [ ]:
# speichere die Datei in einem neuen Shapefile
output_path = Path(data_base_path) / "green_roofs_with_floors.shp"
green_roofs_with_floors.to_file(output_path)

C:\Users\User\AppData\Local\Temp\ipykernel_25076\259876181.py:3: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  green_roofs_with_floors.to_file(output_path)
c:\Users\User\Documents\Uni\Master_Bauing\WiSe25_26\AI_in_Human_Water\berlin-green-roofs\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'gml_id_left' to 'gml_id_lef'
  ogr_write(
c:\Users\User\Documents\Uni\Master_Bauing\WiSe25_26\AI_in_Human_Water\berlin-green-roofs\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'index_right' to 'index_righ'
  ogr_write(
c:\Users\User\Documents\Uni\Master_Bauing\WiSe25_26\AI_in_Human_Water\berlin-green-roofs\.venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'gml_id_right' to 'gml_id_rig'
  ogr_write(
c:\Users\User\Documents\Uni\Master_Bauing\WiSe25_26\AI_in_Human_Water\berlin-green-roofs\.venv\Lib\site-packages

In [11]:
# convert to pandas dataframe to get an attribute table
green_roofs_with_floors_attribute_table = pd.DataFrame(green_roofs_with_floors.drop(columns="geometry"))
print(green_roofs_with_floors_attribute_table.head())

    gml_id_left  importid    geb_nutz  gruendach    ex_int  gruen20_m2  \
0  d_gebaeude.1         1  Tiefgarage  vorhanden  intensiv      210.09   
0  d_gebaeude.1         1  Tiefgarage  vorhanden  intensiv      210.09   
0  d_gebaeude.1         1  Tiefgarage  vorhanden  intensiv      210.09   
0  d_gebaeude.1         1  Tiefgarage  vorhanden  intensiv      210.09   
1  d_gebaeude.2         2  Tiefgarage  vorhanden  intensiv      559.36   

   gint20_m2  gex20_m2  gruen20_p  gint20_p  ...    eex_int  index_right  \
0     207.88      2.21      86.03     85.13  ...  intensive     277022.0   
0     207.88      2.21      86.03     85.13  ...  intensive     926111.0   
0     207.88      2.21      86.03     85.13  ...  intensive     676270.0   
0     207.88      2.21      86.03     85.13  ...  intensive      27181.0   
1     480.07     79.29      61.59     52.86  ...  intensive      29377.0   

                              gml_id_right             gisid  \
0  a_geschosszahl_mehr_10.DEBE06YY

## Lade die fertige Datei ein

In [4]:

# lade die dbf Datei ein
# nutze geopandas um die dbf datei einzulesen, da sie georeferenzierte Informationen enthält
dbf_file_path = Path(data_base_path) / "green_roofs_with_floors.dbf"
dbf_data = gpd.read_file(dbf_file_path)
print(dbf_data.head())
print(dbf_data.columns)
# entferne unnötige spalten, um platz zu sparen
columns_to_drop = ["geometry", "gint20_m2", "gex20_m2", "gruen20_p", "gint20_p",
                   "gruen20_m2", "gml_id_rig", "gisid",  "gmlid", "gfk", "gfk_bezeic", "geschoss_k",]
dbf_data_cleaned = dbf_data.drop(columns=columns_to_drop)
print(dbf_data_cleaned.head())

     gml_id_lef  importid    geb_nutz  gruendach    ex_int  gruen20_m2  \
0  d_gebaeude.1         1  Tiefgarage  vorhanden  intensiv      210.09   
1  d_gebaeude.1         1  Tiefgarage  vorhanden  intensiv      210.09   
2  d_gebaeude.1         1  Tiefgarage  vorhanden  intensiv      210.09   
3  d_gebaeude.1         1  Tiefgarage  vorhanden  intensiv      210.09   
4  d_gebaeude.2         2  Tiefgarage  vorhanden  intensiv      559.36   

   gint20_m2  gex20_m2  gruen20_p  gint20_p  ...  index_righ  \
0     207.88      2.21      86.03     85.13  ...    277022.0   
1     207.88      2.21      86.03     85.13  ...    926111.0   
2     207.88      2.21      86.03     85.13  ...    676270.0   
3     207.88      2.21      86.03     85.13  ...     27181.0   
4     480.07     79.29      61.59     52.86  ...     29377.0   

                                gml_id_rig             gisid  \
0  a_geschosszahl_mehr_10.DEBE06YYA0000835  DEBE06YYA0000835   
1  a_geschosszahl_mehr_10.DEBE06YYA0000835

In [8]:
# entferne doppelte elemente und lösche sie anhand der importid

dbf_data_cleaned = dbf_data_cleaned.drop_duplicates(subset=["importid"])
print(dbf_data_cleaned.head())

#speichere die bereinigte dbf datei als csv datei, um sie später für das maschinelle lernen zu verwenden
output_csv_path = Path(data_base_path) / "green_roofs_with_floors_cleaned.csv"
dbf_data_cleaned.to_csv(output_csv_path, index=False)


      gml_id_lef  importid    geb_nutz        gruendach    ex_int  gex20_p  \
0   d_gebaeude.1         1  Tiefgarage        vorhanden  intensiv     0.90   
4   d_gebaeude.2         2  Tiefgarage        vorhanden  intensiv     8.73   
32  d_gebaeude.3         3  Tiefgarage  nicht vorhanden       NaN     0.00   
36  d_gebaeude.4         4  Tiefgarage        vorhanden  intensiv    23.30   
44  d_gebaeude.5         5  Tiefgarage  nicht vorhanden       NaN     0.00   

    geb_area                nutz  ext            egeb_nutz    egruendach  \
0     244.19  Tiefgarage (ALKIS)    0  Underground parking      existent   
4     908.26  Tiefgarage (ALKIS)    0  Underground parking      existent   
32    435.39  Tiefgarage (ALKIS)    0  Underground parking  non-existent   
36    871.58  Tiefgarage (ALKIS)    0  Underground parking      existent   
44     27.12  Tiefgarage (ALKIS)    0  Underground parking  non-existent   

      eex_int  index_righ  aog  aug  
0   intensive    277022.0  3.0  NaN 